# Solar Park Forecasting: End-to-End Training Pipeline

This notebook contains the actual training workflow for the forecasting project. It starts from the raw split files, constructs the causal features, trains the baseline and machine-learning models, evaluates fresh predictions, and saves the resulting models and metadata under `artifacts/`.

The workflow has three stages:

- **F2:** climatology and CSI persistence baselines.
- **F3:** same-day LightGBM models with power-only and power-plus-weather feature sets.
- **F4:** day-ahead daylight LightGBM, Ridge, and PyTorch MLP models.

All evaluation tables are computed from predictions created during this run. No pre-computed prediction CSV is read for model results.

## 1. Environment and reproducibility

Run this notebook from the package directory. The package includes the raw input tables and the protocol configuration files. A CPU-only environment is sufficient.

In [ ]:
# %%
from pathlib import Path
import json, gzip, math, time, random, warnings
from functools import lru_cache
import numpy as np
import pandas as pd
import joblib
from IPython.display import display
import lightgbm as lgb
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
SEED = 42
SEEDS = [42, 43, 44]
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
    torch.set_num_threads(2)
    torch.manual_seed(SEED)
except ImportError:
    TORCH_AVAILABLE = False
    torch = None

ROOT = Path.cwd()
if not (ROOT / "forecast_inputs").exists():
    ROOT = Path("/home/ubuntu/work_forecast/forecasting_training_pipeline")
INPUT_DIR = ROOT / "forecast_inputs"
ARTIFACT_DIR = ROOT / "artifacts"
MODEL_DIR = ARTIFACT_DIR / "models"
for p in [ARTIFACT_DIR, MODEL_DIR, ARTIFACT_DIR / "f3_models", ARTIFACT_DIR / "f4_models"]:
    p.mkdir(parents=True, exist_ok=True)

F2_CONFIG = json.loads((ROOT / "f2_config.json").read_text())
F3_CONFIG = json.loads((ROOT / "f3_config.json").read_text())
F4_CONFIG = json.loads((ROOT / "f4_config.json").read_text())

print("Project root:", ROOT)
print("PyTorch available:", TORCH_AVAILABLE)


## 2. Load raw data and define the split masks

The three split directories are joined on `time_utc`. Quality flags, warmup rows, purge-zone rows, and plant-zero events are handled before any feature or target is built. Training-only maps are constructed later from the cleaned training split.

In [ ]:
# %%
def load_split(split):
    target = pd.read_csv(INPUT_DIR / f"{split}_targets.csv.gz", parse_dates=["time_utc"]).set_index("time_utc")
    quality = pd.read_csv(INPUT_DIR / f"{split}_data_quality_flags.csv.gz", parse_dates=["time_utc"]).set_index("time_utc")
    weather = pd.read_csv(INPUT_DIR / f"{split}_features_weather.csv.gz", parse_dates=["time_utc"]).set_index("time_utc")
    quality = quality.drop(columns=["split"], errors="ignore")
    out = target.join(quality, how="left").join(weather, how="left")
    out["split"] = split
    return out

parts = [load_split(s) for s in ["train", "val", "test"]]
df = pd.concat(parts).sort_index()
df = df.loc[:, ~df.columns.duplicated()]
df = df[~df.index.duplicated(keep="last")]

events = pd.read_csv(INPUT_DIR / "events.csv", parse_dates=["start_utc", "end_utc"])
purge = pd.Series(False, index=df.index)
for _, event in events[events["type"].eq("plant_zero_daytime")].iterrows():
    purge |= (
        (df.index >= event.start_utc - pd.Timedelta(hours=1)) &
        (df.index <= event.end_utc + pd.Timedelta(hours=1))
    )

df["purged"] = purge.to_numpy()
df["train_ok"] = (
    df["split"].eq("train") & df["is_day"].eq(1) & df["plant_pu"].notna()
    & ~df["purged"] & ~df["in_purge_zone"].eq(1) & df["is_warmup"].eq(0)
)
df["eval_ok"] = (
    df["split"].isin(["val", "test"]) & df["is_day"].eq(1) & df["plant_pu"].notna()
    & ~df["purged"] & ~df["in_purge_zone"].eq(1) & df["is_warmup"].eq(0)
)
print(df.groupby("split").size().rename("rows"))
print(pd.DataFrame({
    "train_ok": df.groupby("split")["train_ok"].sum(),
    "eval_ok": df.groupby("split")["eval_ok"].sum(),
}))


## 3. Train-only climatology, environment map, and causal lookup functions

The environment map is the 95th percentile of training power for each circular day-of-year and 15-minute time bin, using a ±15-day training window. The climatology is the training mean for the same calendar bin. These maps never use validation or test targets.

In [ ]:
# %%
df["doy"] = ((df.index.dayofyear - 1) % 365) + 1
df["hm"] = df.index.hour * 60 + df.index.minute
train = df[df["train_ok"]].copy()

climatology = train.groupby(["doy", "hm"])["plant_pu"].mean().to_dict()
env_frames = []
for shift in range(-15, 16):
    z = train[["doy", "hm", "plant_pu"]].copy()
    z["target_doy"] = ((z["doy"] - 1 + shift) % 365) + 1
    env_frames.append(z[["target_doy", "hm", "plant_pu"]])
env_series = pd.concat(env_frames).groupby(["target_doy", "hm"])["plant_pu"].quantile(0.95)
environment = {(int(d), int(hm)): float(v) for (d, hm), v in env_series.items()}
y_map = df["plant_pu"].to_dict()
weather_maps = {c: df[c].to_dict() for c in ["irr_wm2", "temp_c", "mod_temp_c", "solar_elev_deg"]}

def env_at(t):
    return environment.get((int(((t.dayofyear - 1) % 365) + 1), int(t.hour * 60 + t.minute)), np.nan)

def clim_at(t):
    return climatology.get((int(((t.dayofyear - 1) % 365) + 1), int(t.hour * 60 + t.minute)), np.nan)

def y_at(t):
    return y_map.get(t, np.nan)

def csi_at(t):
    e, y = env_at(t), y_at(t)
    return y / e if np.isfinite(e) and e >= 0.05 and np.isfinite(y) else np.nan

def safe_mean(values):
    values = np.asarray(values, dtype=float)
    return float(np.nanmean(values)) if np.isfinite(values).any() else np.nan

def safe_std(values):
    values = np.asarray(values, dtype=float)
    return float(np.nanstd(values, ddof=1)) if np.isfinite(values).sum() > 1 else np.nan

train_days = train.index.floor("D")
train_day_csi = pd.Series([csi_at(t) for t in train.index], index=train.index)
daily_csi_map = train_day_csi.groupby(train_days).mean().to_dict()

def daily_csi(o, k):
    vals = [daily_csi_map.get(o.floor("D") - pd.Timedelta(days=j), np.nan) for j in range(1, k + 1)]
    return safe_mean(vals)

def last_day_same_time(t, k):
    return safe_mean([csi_at(t - pd.Timedelta(days=j)) for j in range(1, k + 1)])

with open(ARTIFACT_DIR / "train_only_maps.json", "w") as f:
    json.dump({"climatology": {str(k): v for k, v in climatology.items()},
               "environment": {str(k): v for k, v in environment.items()}}, f)
print("Training map bins:", len(climatology), "climatology |", len(environment), "environment")


## 4. F2 baselines: climatology and persistence

F2 baselines are computed directly from the target timestamps. The persistence variants use the mean CSI from the previous 1, 3, or 7 same-time days, then convert CSI back to power with the target-time environment map. The results are produced from the raw data in this run.

In [ ]:
# %%
def horizon_from_target(t):
    return 192 if (t.hour == 0 and t.minute == 0) else t.hour * 4 + t.minute // 15 + 96

def subset_name(split, target):
    if split == "val":
        return "march_2022" if target.month == 3 else "val_no_march_2022"
    if split == "test":
        return "test_before_2023-10-12" if target < pd.Timestamp("2023-10-12", tz="UTC") else "test_after_2023-10-12"
    return "train"

f2_records = []
for split in ["val", "test"]:
    targets = df[(df["split"] == split) & df["eval_ok"]]
    for t, row in targets.iterrows():
        e = env_at(t)
        clim = clim_at(t)
        if not np.isfinite(e) or e < 0.05 or not np.isfinite(row.plant_pu):
            continue
        candidates = {
            "climatology": clim,
            "csi_lastk_1d": e * np.clip(last_day_same_time(t, 1), 0, 1.3),
            "csi_lastk_3d": e * np.clip(last_day_same_time(t, 3), 0, 1.3),
            "csi_lastk_7d": e * np.clip(last_day_same_time(t, 7), 0, 1.3),
        }
        h = horizon_from_target(t)
        h_group = "h<=96" if h <= 96 else "h=97..192"
        for baseline, pred in candidates.items():
            if baseline == "climatology": pred = clim
            f2_records.append({"split": split, "subset": subset_name(split, t),
                               "baseline": baseline, "target": t, "y": row.plant_pu,
                               "pred": pred, "h": h, "horizon_group": h_group})
f2_pred = pd.DataFrame(f2_records)

def metric_row(group):
    y, p = group.y.to_numpy(float), group.pred.to_numpy(float)
    return pd.Series({"MAE": mean_absolute_error(y, p), "RMSE": mean_squared_error(y, p, squared=False),
                      "R2": r2_score(y, p), "n": len(group), "n_days": group.target.dt.floor("D").nunique()})

f2_metrics = f2_pred.groupby(["split", "subset", "baseline", "horizon_group"], dropna=False).apply(metric_row, include_groups=False).reset_index()
clim_rmse = f2_metrics[f2_metrics.baseline.eq("climatology")].set_index(["split", "subset", "horizon_group"])["RMSE"]
f2_metrics["skill"] = f2_metrics.apply(lambda r: 1 - r.RMSE / clim_rmse.get((r.split, r.subset, r.horizon_group), np.nan), axis=1)
f2_metrics.to_csv(ARTIFACT_DIR / "f2_metrics.csv", index=False)
f2_pred.to_csv(ARTIFACT_DIR / "f2_predictions.csv.gz", index=False, compression="gzip")
display(f2_metrics.sort_values(["split", "subset", "RMSE"]))


## 5. F3 feature construction

F3 creates examples for horizons 4, 8, 16, and 24 fifteen-minute steps. Features are available at the forecast origin, while the target is the future clipped CSI. `PWR+WX` adds weather available at the origin.

In [ ]:
# %%
F3_HORIZONS = [4, 8, 16, 24]
F3_FEATURE_SETS = ["PWR", "PWR+WX"]

def f3_feature_row(origin, target, feature_set):
    csi_history = [csi_at(origin - pd.Timedelta(minutes=15 * j)) for j in range(1, 17)][::-1]
    values = {
        "csi_o": csi_history[-1], "csi_o_m1": csi_history[-2], "csi_o_m4": csi_history[-5],
        "csi_mean4": safe_mean(csi_history[-4:]), "csi_mean8": safe_mean(csi_history[-8:]),
        "csi_mean16": safe_mean(csi_history), "csi_std8": safe_std(csi_history[-8:]),
        "y_o": y_at(origin), "y_o_m4": y_at(origin - pd.Timedelta(minutes=60)),
        "env_o": env_at(origin), "env_t": env_at(target),
        "clim_over_env": clim_at(target) / env_at(target) if np.isfinite(env_at(target)) and env_at(target) >= 0.05 else np.nan,
        "mean7_over_env": safe_mean([y_at(target - pd.Timedelta(days=j)) for j in range(1, 8)]) / env_at(target) if np.isfinite(env_at(target)) and env_at(target) >= 0.05 else np.nan,
        "daily_csi_1d": daily_csi(origin, 1), "daily_csi_3d": daily_csi(origin, 3),
        "solar_o": weather_maps["solar_elev_deg"].get(origin, np.nan),
        "solar_t": weather_maps["solar_elev_deg"].get(target, np.nan),
        "hour_sin": np.sin(2 * np.pi * target.hour / 24), "hour_cos": np.cos(2 * np.pi * target.hour / 24),
        "doy_sin": np.sin(2 * np.pi * (((target.dayofyear - 1) % 365) + 1) / 365),
        "doy_cos": np.cos(2 * np.pi * (((target.dayofyear - 1) % 365) + 1) / 365),
    }
    if feature_set == "PWR+WX":
        values.update({
            "irr_o": weather_maps["irr_wm2"].get(origin, np.nan),
            "temp_o": weather_maps["temp_c"].get(origin, np.nan),
            "mod_temp_o": weather_maps["mod_temp_c"].get(origin, np.nan),
            "irr_mean4": safe_mean([weather_maps["irr_wm2"].get(origin - pd.Timedelta(minutes=15 * j), np.nan) for j in range(1, 5)]),
        })
    return values

f3_examples = {}
for h in F3_HORIZONS:
    for feature_set in F3_FEATURE_SETS:
        records = []
        for split in ["train", "val", "test"]:
            origins = df[(df["split"] == split) & df["is_day"].eq(1) & df["is_warmup"].eq(0) & ~df["in_purge_zone"].eq(1)].index
            target_map = df[df["train_ok"]].index if split == "train" else df[df["eval_ok"]].index
            target_map = set(target_map)
            for origin in origins:
                target = origin + pd.Timedelta(minutes=15 * h)
                if target not in target_map:
                    continue
                e, c = env_at(target), csi_at(target)
                if not np.isfinite(e) or e < 0.05 or not np.isfinite(c):
                    continue
                row = f3_feature_row(origin, target, feature_set)
                row.update({"origin": origin, "target": target, "split": split,
                            "y": y_at(target), "csi_target": np.clip(c, 0, 1.3)})
                records.append(row)
        f3_examples[h, feature_set] = pd.DataFrame(records)
        print(h, feature_set, f3_examples[h, feature_set].shape)


## 6. F3 LightGBM training, early stopping, and hyperparameter search

For each horizon and feature set, a small validation search selects the number of leaves and minimum leaf size. The selected configuration is then retrained for seeds 42, 43, and 44 with early stopping on validation days outside March 2022.

In [ ]:
# %%
F3_TUNE = [(15, 100), (15, 200), (31, 100), (31, 200)]
f3_tuning_rows, f3_predictions = [], []
f3_models = {}
f3_importance = []

for h in F3_HORIZONS:
    for feature_set in F3_FEATURE_SETS:
        data = f3_examples[h, feature_set].reset_index(drop=True)
        feature_cols = [c for c in data.columns if c not in ["origin", "target", "split", "y", "csi_target"]]
        train_mask = data.split.eq("train")
        val_mask = data.split.eq("val") & data.target.dt.month.ne(3)
        X_train, y_train = data.loc[train_mask, feature_cols], data.loc[train_mask, "csi_target"]
        X_val, y_val = data.loc[val_mask, feature_cols], data.loc[val_mask, "csi_target"]
        best = None
        for leaves, min_leaf in F3_TUNE:
            model = lgb.LGBMRegressor(objective="regression_l2", learning_rate=0.05,
                num_leaves=leaves, min_data_in_leaf=min_leaf, feature_fraction=0.8,
                bagging_fraction=0.8, bagging_freq=1, n_estimators=2000,
                random_state=SEED, verbosity=-1)
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="rmse",
                      callbacks=[lgb.early_stopping(50, verbose=False)])
            rmse = mean_squared_error(y_val, model.predict(X_val), squared=False)
            row = {"horizon": h, "feature_set": feature_set, "num_leaves": leaves,
                   "min_data_in_leaf": min_leaf, "val_RMSE": rmse, "best_iteration": model.best_iteration_}
            f3_tuning_rows.append(row)
            if best is None or rmse < best["val_RMSE"]:
                best = row
        for seed in SEEDS:
            model = lgb.LGBMRegressor(objective="regression_l2", learning_rate=0.05,
                num_leaves=int(best["num_leaves"]), min_data_in_leaf=int(best["min_data_in_leaf"]),
                feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
                n_estimators=2000, random_state=seed, verbosity=-1)
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="rmse",
                      callbacks=[lgb.early_stopping(50, verbose=False)])
            f3_models[h, feature_set, seed] = (model, feature_cols)
            model.booster_.save_model(str(ARTIFACT_DIR / "f3_models" / f"f3_h{h}_{feature_set.replace('+','plus')}_seed{seed}.txt"))
            gain = model.booster_.feature_importance("gain")
            total = gain.sum()
            for name, value in zip(feature_cols, gain):
                f3_importance.append({"horizon": h, "feature_set": feature_set, "feature": name,
                                      "gain": value, "gain_pct": 100 * value / total if total else 0, "seed": seed})
            for split in ["val", "test"]:
                mask = data.split.eq(split)
                q = data.loc[mask].copy()
                csi_pred = model.predict(q[feature_cols])
                power_pred = q["target"].map(env_at).to_numpy() * np.clip(csi_pred, 0, 1.3)
                for target, y, pred in zip(q.target, q.y, power_pred):
                    f3_predictions.append({"horizon": h, "feature_set": feature_set, "seed": seed,
                        "split": split, "subset": subset_name(split, target), "target": target,
                        "y": y, "pred": pred})

f3_tuning = pd.DataFrame(f3_tuning_rows)
f3_pred = pd.DataFrame(f3_predictions)
f3_imp = pd.DataFrame(f3_importance)
f3_tuning.to_csv(ARTIFACT_DIR / "f3_tuning.csv", index=False)
f3_pred.to_csv(ARTIFACT_DIR / "f3_predictions.csv.gz", index=False, compression="gzip")
f3_imp.to_csv(ARTIFACT_DIR / "f3_feature_importance.csv", index=False)
display(f3_tuning.sort_values(["horizon", "feature_set", "val_RMSE"]).groupby(["horizon", "feature_set"]).head(1))


## 7. F3 dynamic evaluation

Metrics are calculated from the predictions generated in the previous cell. The baseline anchor is the minimum non-climatology F2 RMSE for the matching split and subset.

In [ ]:
# %%
def summarize_predictions(predictions, group_cols):
    rows = []
    for keys, group in predictions.groupby(group_cols):
        y, p = group.y.to_numpy(float), group.pred.to_numpy(float)
        row = dict(zip(group_cols, keys if isinstance(keys, tuple) else [keys]))
        row.update({"MAE": mean_absolute_error(y, p), "RMSE": mean_squared_error(y, p, squared=False),
                    "R2": r2_score(y, p), "n": len(group),
                    "n_days": group.target.dt.floor("D").nunique()})
        rows.append(row)
    return pd.DataFrame(rows)

f3_metrics = summarize_predictions(f3_pred, ["horizon", "feature_set", "split", "subset"])
anchor = f2_metrics[(f2_metrics.baseline != "climatology")].groupby(["split", "subset"])["RMSE"].min()
f3_metrics["anchor_RMSE"] = [anchor.get((r.split, r.subset), np.nan) for _, r in f3_metrics.iterrows()]
f3_metrics["skill_bar"] = 1 - f3_metrics.RMSE / f3_metrics.anchor_RMSE
f3_metrics.to_csv(ARTIFACT_DIR / "f3_metrics.csv", index=False)
display(f3_metrics.sort_values(["subset", "horizon", "feature_set"]))


## 8. F4 day-ahead feature construction

F4 uses daily UTC origins and horizons 97–192. The corrected `same_time_*` features query historical CSI at the daylight target time, not at the midnight origin. The training examples below retain only valid daylight targets with `env_t >= 0.05`.

In [ ]:
# %%
F4_HORIZONS = list(range(97, 193))
F4_FEATURES = ["h", "hour_sin", "hour_cos", "doy_sin", "doy_cos", "solar_t", "env_t",
    "clim_over_env", "mean7_over_env", "daily_csi_1d", "daily_csi_2d", "daily_csi_3d",
    "daily_csi_5d", "daily_csi_7d", "daily_csi_14d", "daily_csi_std7", "same_time_1d",
    "same_time_3d", "same_time_7d", "csi_mean8_isday"]

day_times = train.index[train["is_day"].eq(1)]
day_csi = pd.Series([csi_at(t) for t in day_times], index=day_times)
day_csi_values = day_csi.to_numpy(float)
day_time_values = day_times.astype("int64").to_numpy()

@lru_cache(maxsize=None)
def f4_row(origin, target, h):
    e = env_at(target)
    clim_over_env = clim_at(target) / e if np.isfinite(e) and e >= 0.05 else np.nan
    mean7_over_env = safe_mean([y_at(target - pd.Timedelta(days=j)) for j in range(1, 8)]) / e if np.isfinite(e) and e >= 0.05 else np.nan
    position = np.searchsorted(day_time_values, origin.value, side="left")
    recent = day_csi_values[max(0, position - 8):position]
    doy = ((target.dayofyear - 1) % 365) + 1
    return {
        "h": int(h), "hour_sin": np.sin(2 * np.pi * target.hour / 24), "hour_cos": np.cos(2 * np.pi * target.hour / 24),
        "doy_sin": np.sin(2 * np.pi * doy / 365), "doy_cos": np.cos(2 * np.pi * doy / 365),
        "solar_t": weather_maps["solar_elev_deg"].get(target, np.nan), "env_t": e,
        "clim_over_env": clim_over_env, "mean7_over_env": mean7_over_env,
        "daily_csi_1d": daily_csi(origin, 1), "daily_csi_2d": daily_csi(origin, 2),
        "daily_csi_3d": daily_csi(origin, 3), "daily_csi_5d": daily_csi(origin, 5),
        "daily_csi_7d": daily_csi(origin, 7), "daily_csi_14d": daily_csi(origin, 14),
        "daily_csi_std7": safe_std([daily_csi_map.get(origin.floor("D") - pd.Timedelta(days=j), np.nan) for j in range(1, 8)]),
        "same_time_1d": last_day_same_time(target, 1), "same_time_3d": last_day_same_time(target, 3),
        "same_time_7d": last_day_same_time(target, 7), "csi_mean8_isday": safe_mean(recent),
    }

def build_f4_split(split):
    records = []
    origins = sorted(pd.DatetimeIndex(df[(df["split"] == split) & (df["is_day"] == 1) & (~df["in_purge_zone"].eq(1)) & (~df["is_warmup"].eq(1))].index).normalize().unique())
    valid_targets = set(df[df["train_ok"] if split == "train" else df["eval_ok"]].index)
    for origin_day in origins:
        origin = pd.Timestamp(origin_day)
        if origin not in df.index:
            continue
        for h in F4_HORIZONS:
            target = origin + pd.Timedelta(minutes=15 * h)
            if target not in valid_targets:
                continue
            e, c = env_at(target), csi_at(target)
            if not np.isfinite(e) or e < 0.05 or not np.isfinite(c):
                continue
            row = f4_row(origin, target, h)
            row.update({"origin": origin, "target": target, "split": split,
                        "y": y_at(target), "csi_target": np.clip(c, 0, 1.3)})
            records.append(row)
    return pd.DataFrame(records)

f4_data = {split: build_f4_split(split) for split in ["train", "val", "test"]}
f4_all = pd.concat(f4_data.values(), ignore_index=True)
print({k: v.shape for k, v in f4_data.items()})


## 9. F4 hyperparameter search and model training

LightGBM uses early stopping on validation days outside March 2022. Ridge uses median imputation and standard scaling. The MLP uses a PyTorch regression loop with Adam, mean-squared error loss, validation early stopping, and the same scaled feature matrix. Each family is trained for seeds 42, 43, and 44.

In [ ]:
# %%
if not TORCH_AVAILABLE:
    raise ImportError("PyTorch is required for the F4 MLP. Install requirements.txt before running the notebook.")

F4_TUNE = {
    "LightGBM": [(7, 300), (15, 300), (31, 300), (31, 600)],
    "Ridge": [0.1, 1.0, 10.0, 100.0, 1000.0],
    "MLP": [((64, 32), 1e-4), ((64, 32), 1e-2), ((128, 64), 1e-4), ((128, 64), 1e-2)],
}
train4, val4 = f4_data["train"], f4_data["val"]
val4_main = val4[val4.target.dt.month.ne(3)]
X_train4, y_train4 = train4[F4_FEATURES], train4.csi_target
X_val4, y_val4 = val4_main[F4_FEATURES], val4_main.csi_target
imputer4 = SimpleImputer(strategy="median")
scaler4 = StandardScaler()
X_train4_scaled = scaler4.fit_transform(imputer4.fit_transform(X_train4))
X_val4_scaled = scaler4.transform(imputer4.transform(X_val4))
joblib.dump(imputer4, ARTIFACT_DIR / "f4_imputer.joblib")
joblib.dump(scaler4, ARTIFACT_DIR / "f4_scaler.joblib")

f4_tuning = []
# LightGBM search
for leaves, min_leaf in F4_TUNE["LightGBM"]:
    model = lgb.LGBMRegressor(objective="regression_l2", learning_rate=0.05,
        num_leaves=leaves, min_data_in_leaf=min_leaf, feature_fraction=0.8,
        bagging_fraction=0.8, bagging_freq=1, n_estimators=2000, random_state=SEED, verbosity=-1)
    model.fit(X_train4, y_train4, eval_set=[(X_val4, y_val4)], eval_metric="rmse",
              callbacks=[lgb.early_stopping(50, verbose=False)])
    f4_tuning.append({"family": "LightGBM", "settings": f"leaves={leaves},min_leaf={min_leaf}",
                      "val_RMSE_CSI": mean_squared_error(y_val4, model.predict(X_val4), squared=False),
                      "best_iteration": model.best_iteration_})

# Ridge search
for alpha in F4_TUNE["Ridge"]:
    model = Ridge(alpha=alpha).fit(X_train4_scaled, y_train4)
    f4_tuning.append({"family": "Ridge", "settings": f"alpha={alpha}",
                      "val_RMSE_CSI": mean_squared_error(y_val4, model.predict(X_val4_scaled), squared=False),
                      "best_iteration": 0})

# MLP search uses one seeded PyTorch fit per candidate.
def fit_torch_mlp(X_train, y_train, X_val, y_val, hidden, weight_decay, seed, epochs=80, patience=8):
    if not TORCH_AVAILABLE:
        raise ImportError("PyTorch is required for the F4 MLP. Install requirements.txt before running the notebook.")
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(X_train.shape[1], hidden[0]), nn.ReLU(),
                          nn.Linear(hidden[0], hidden[1]), nn.ReLU(), nn.Linear(hidden[1], 1))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1))
    loader = DataLoader(train_ds, batch_size=512, shuffle=True)
    Xv = torch.tensor(X_val, dtype=torch.float32)
    yv = torch.tensor(y_val.to_numpy(), dtype=torch.float32).reshape(-1, 1)
    best_state, best_loss, wait = None, np.inf, 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad(): val_loss = float(loss_fn(model(Xv), yv))
        if val_loss < best_loss - 1e-7:
            best_loss, best_state, wait = val_loss, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state)
    return model, best_loss, epoch + 1

if TORCH_AVAILABLE:
    for hidden, weight_decay in F4_TUNE["MLP"]:
        _, val_loss, epochs = fit_torch_mlp(X_train4_scaled, y_train4, X_val4_scaled, y_val4, hidden, weight_decay, SEED)
        f4_tuning.append({"family": "MLP", "settings": f"hidden={hidden},weight_decay={weight_decay}",
                          "val_RMSE_CSI": np.sqrt(val_loss), "best_iteration": epochs})

f4_tuning_df = pd.DataFrame(f4_tuning)
f4_tuning_df.to_csv(ARTIFACT_DIR / "f4_tuning.csv", index=False)
winners = {family: f4_tuning_df[f4_tuning_df.family.eq(family)].sort_values("val_RMSE_CSI").iloc[0].to_dict()
           for family in f4_tuning_df.family.unique()}
display(f4_tuning_df.sort_values(["family", "val_RMSE_CSI"]).groupby("family").head(1))


## 10. Fit final F4 models, save artifacts, and evaluate fresh predictions

The selected configuration for each family is trained independently for each seed. Predictions are generated immediately from the in-memory models and used for all reported metrics. Model files, scalers, tuning logs, predictions, and configuration metadata are written to `artifacts/`.

In [ ]:
# %%
def parse_lgbm_settings(settings):
    parts = dict(item.split("=") for item in settings.split(","))
    return int(parts["leaves"]), int(parts["min_leaf"])

def parse_mlp_settings(settings):
    hidden_text = settings.split("hidden=")[1].split(",weight_decay")[0].strip("()")
    hidden = tuple(int(x.strip()) for x in hidden_text.split(",") if x.strip())
    wd = float(settings.split("weight_decay=")[1])
    return hidden, wd

def csi_to_power(pred_csi, targets):
    env = targets.map(env_at).to_numpy(float)
    return env * np.clip(np.asarray(pred_csi, dtype=float), 0, 1.3)

f4_predictions = []
f4_importance = []
for family, winner in winners.items():
    for seed in SEEDS:
        if family == "LightGBM":
            leaves, min_leaf = parse_lgbm_settings(winner["settings"])
            model = lgb.LGBMRegressor(objective="regression_l2", learning_rate=0.05,
                num_leaves=leaves, min_data_in_leaf=min_leaf, feature_fraction=0.8,
                bagging_fraction=0.8, bagging_freq=1, n_estimators=2000,
                random_state=seed, verbosity=-1)
            model.fit(X_train4, y_train4, eval_set=[(X_val4, y_val4)], eval_metric="rmse",
                      callbacks=[lgb.early_stopping(50, verbose=False)])
            model.booster_.save_model(str(ARTIFACT_DIR / "f4_models" / f"lgbm_seed{seed}.txt"))
            gain = model.booster_.feature_importance("gain")
            total = gain.sum()
            for name, value in zip(F4_FEATURES, gain):
                f4_importance.append({"family": family, "seed": seed, "feature": name, "gain": value,
                                      "gain_pct": 100 * value / total if total else 0})
            def predict_frame(frame): return model.predict(frame[F4_FEATURES])
        elif family == "Ridge":
            alpha = float(winner["settings"].split("=")[1])
            model = Ridge(alpha=alpha).fit(X_train4_scaled, y_train4)
            joblib.dump(model, ARTIFACT_DIR / "f4_models" / f"ridge_seed{seed}.joblib")
            def predict_frame(frame): return model.predict(scaler4.transform(imputer4.transform(frame[F4_FEATURES])))
        elif family == "MLP":
            hidden, weight_decay = parse_mlp_settings(winner["settings"])
            model, _, epochs = fit_torch_mlp(X_train4_scaled, y_train4, X_val4_scaled, y_val4, hidden, weight_decay, seed)
            torch.save({"state_dict": model.state_dict(), "hidden": hidden, "features": F4_FEATURES,
                        "epochs": epochs, "weight_decay": weight_decay}, ARTIFACT_DIR / "f4_models" / f"mlp_seed{seed}.pt")
            def predict_frame(frame):
                xt = torch.tensor(scaler4.transform(imputer4.transform(frame[F4_FEATURES])), dtype=torch.float32)
                model.eval()
                with torch.no_grad(): return model(xt).numpy().ravel()
        else:
            continue
        for split, frame in f4_data.items():
            csi_pred = predict_frame(frame)
            power_pred = csi_to_power(csi_pred, frame.target)
            for target, y, pred in zip(frame.target, frame.y, power_pred):
                f4_predictions.append({"family": family, "seed": seed, "split": split,
                    "subset": subset_name(split, target), "target": target, "y": y, "pred": pred})

f4_pred = pd.DataFrame(f4_predictions)
f4_imp = pd.DataFrame(f4_importance)
f4_pred.to_csv(ARTIFACT_DIR / "f4_predictions.csv.gz", index=False, compression="gzip")
f4_imp.to_csv(ARTIFACT_DIR / "f4_feature_importance.csv", index=False)


In [ ]:
# %%
f4_metrics = summarize_predictions(f4_pred, ["family", "seed", "split", "subset"])
anchor_f4 = f2_metrics[(f2_metrics.baseline != "climatology")].groupby(["split", "subset"])["RMSE"].min()
f4_metrics["anchor_RMSE"] = [anchor_f4.get((r.split, r.subset), np.nan) for _, r in f4_metrics.iterrows()]
f4_metrics["skill_bar"] = 1 - f4_metrics.RMSE / f4_metrics.anchor_RMSE
f4_metrics.to_csv(ARTIFACT_DIR / "f4_metrics.csv", index=False)

f4_summary = (f4_metrics.groupby(["family", "split", "subset"], as_index=False)
              .agg(MAE=("MAE", "mean"), RMSE=("RMSE", "mean"), R2=("R2", "mean"),
                   RMSE_sd=("RMSE", "std"), n=("n", "first"), anchor_RMSE=("anchor_RMSE", "first")))
f4_summary["skill_bar"] = 1 - f4_summary.RMSE / f4_summary.anchor_RMSE
f4_summary.to_csv(ARTIFACT_DIR / "f4_summary.csv", index=False)
display(f4_summary.sort_values(["subset", "RMSE"]))

run_config = {
    "seed": SEED, "seeds": SEEDS, "f2": F2_CONFIG, "f3": F3_CONFIG, "f4": F4_CONFIG,
    "f3_horizons": F3_HORIZONS, "f3_feature_sets": F3_FEATURE_SETS, "f4_horizons": [97, 192],
    "artifacts": sorted(str(p.relative_to(ARTIFACT_DIR)) for p in ARTIFACT_DIR.rglob("*") if p.is_file()),
}
(ARTIFACT_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2, default=str))
print("Saved artifacts:", len(run_config["artifacts"]))


## 11. Final checks

The checks below confirm that the run produced model files, scalers, dynamic metrics, and the required split/subset dimensions.

In [ ]:
# %%
required_files = [
    ARTIFACT_DIR / "f2_metrics.csv", ARTIFACT_DIR / "f3_metrics.csv", ARTIFACT_DIR / "f4_summary.csv",
    ARTIFACT_DIR / "f4_imputer.joblib", ARTIFACT_DIR / "f4_scaler.joblib", ARTIFACT_DIR / "run_config.json",
]
assert all(p.exists() for p in required_files), [str(p) for p in required_files if not p.exists()]
assert f3_pred["pred"].notna().all() and f4_pred["pred"].notna().all()
assert set(f4_summary["family"]) == set(winners)
print("Training pipeline checks passed.")
print("F2 metric rows:", len(f2_metrics))
print("F3 metric rows:", len(f3_metrics))
print("F4 summary rows:", len(f4_summary))
print("Model files:", len(list((ARTIFACT_DIR / "f3_models").glob("*"))) + len(list((ARTIFACT_DIR / "f4_models").glob("*"))))
